In [2]:
import pandas as pd
from pathlib import Path

base_results = Path(
    "/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/results"
)

paths = {
    "sf0.1": base_results / "ldbc_snb_sf0_1_full_fiben_format_clean" / "schemalens_reduction_analysis_hot.csv",
    "sf1": base_results / "ldbc_snb_sf1_full_fiben_format_clean" / "schemalens_reduction_analysis_hot.csv",
    "sf3": base_results / "ldbc_snb_sf3_full_fiben_format_clean" / "schemalens_reduction_analysis_hot.csv",
}

dfs = []

for scale, path in paths.items():
    df = pd.read_csv(path)
    df["scale_label"] = scale
    dfs.append(df)

cross_scale_schemalens_df = pd.concat(dfs, ignore_index=True)

summary_df = (
    cross_scale_schemalens_df
    .groupby("scale_label")
    .agg(
        n_queries=("query_name", "nunique"),
        avg_DSR=("DSR", "mean"),
        top1_preservation=("top1_preserved_by_activated", "mean"),
        mean_activated_regret=("activated_regret", "mean"),
        mean_primary_regret=("primary_regret", lambda x: x.dropna().mean()),
        primary_winners=("best_group", lambda x: int((x == "primary").sum())),
        secondary_winners=("best_group", lambda x: int((x == "secondary_affected").sum())),
        control_winners=("best_group", lambda x: int((x == "control").sum())),
    )
    .reset_index()
)

display(summary_df)

secondary_df = cross_scale_schemalens_df[
    cross_scale_schemalens_df["best_group"] == "secondary_affected"
][
    [
        "scale_label",
        "official_id",
        "query_name",
        "best_config",
        "best_design_pattern",
        "best_p95_ms",
        "best_primary_config",
        "best_primary_p95_ms",
        "primary_regret",
    ]
].sort_values(["scale_label", "official_id"])

display(secondary_df)

control_df = cross_scale_schemalens_df[
    cross_scale_schemalens_df["best_group"] == "control"
][
    [
        "scale_label",
        "official_id",
        "query_name",
        "best_config",
        "best_design_pattern",
        "best_p95_ms",
        "best_primary_config",
        "best_primary_p95_ms",
        "primary_regret",
        "activated_regret",
    ]
]

display(control_df)

out_dir = base_results / "cross_scale_analysis_final"
out_dir.mkdir(parents=True, exist_ok=True)

cross_scale_schemalens_df.to_csv(out_dir / "ldbc_snb_cross_scale_schemalens_analysis_final.csv", index=False)
summary_df.to_csv(out_dir / "ldbc_snb_cross_scale_summary_final.csv", index=False)
secondary_df.to_csv(out_dir / "ldbc_snb_cross_scale_secondary_winners_final.csv", index=False)
control_df.to_csv(out_dir / "ldbc_snb_cross_scale_control_winners_final.csv", index=False)

print("Saved to:", out_dir)

,scale_label,n_queries,avg_DSR,top1_preservation,mean_activated_regret,mean_primary_regret,primary_winners,secondary_winners,control_winners
0,sf0.1,22,0.713636,1.000000,0.000000,0.025313,19,3,0
1,sf1,22,0.713636,0.954545,0.023537,0.033819,15,6,1
2,sf3,22,0.713636,1.000000,0.000000,0.089429,15,7,0


,scale_label,official_id,query_name,best_config,best_design_pattern,best_p95_ms,best_primary_config,best_primary_p95_ms,primary_regret
4,sf0.1,IC5,IC5_NewGroups,G7,containment_baseline,135.322968,G0,158.656699,0.172430
6,sf0.1,IC7,IC7_RecentLikers,G4,explicit_edge_collection,7.464429,G0,10.145230,0.359143
16,sf0.1,IS2,IS2_RecentMessagesOfPerson,G6,referenced_or_reverse_indexed_edges,2.255789,NaN,NaN,NaN
24,sf1,IC3,IC3_FriendsAndFriendsOfFriendsInCountries,G7,containment_baseline,191.817495,G3,196.627160,0.025074
26,sf1,IC5,IC5_NewGroups,G6,referenced_or_reverse_indexed_edges,176.753622,G0,185.673138,0.050463
34,sf1,INS6,INS6_AddPost,G9,hybrid_containment,2.188292,G3,2.343452,0.070905
35,sf1,INS7,INS7_AddComment,G9,hybrid_containment,2.159610,G3,2.172902,0.006154
38,sf1,IS2,IS2_RecentMessagesOfPerson,G9,hybrid_containment,3.230855,NaN,NaN,NaN
43,sf1,IS7,IS7_RepliesOfMessage,G9,hybrid_containment,11.009079,G0,11.447059,0.039784
48,sf3,IC5,IC5_NewGroups,G7,containment_baseline,193.801405,G3,205.203150,0.058832


,scale_label,official_id,query_name,best_config,best_design_pattern,best_p95_ms,best_primary_config,best_primary_p95_ms,primary_regret,activated_regret
40,sf1,IS4,IS4_ContentOfMessage,G0,root_with_references,2.288074,G1,3.472862,0.51781,0.51781


Saved to: /home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/results/cross_scale_analysis_final
